# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SUKRIT004/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Can search-performance and content signals identify visible pages that deserve CTR/engagement review?

**The decision this supports:** which pages should a content/SEO team review first for possible CTR or engagement improvement? **The action:** rank pages into a review queue with a transparent reason for why each page was prioritized. **What this is not:** proof of Google's ranking algorithm, and not causal proof that changing a page improves CTR or rankings — this is decision support, built and validated on one anonymized dataset.

In [1]:
# Section 1 has no computation of its own — the question is framed in words above.
print("Research question: Can search-performance and content signals identify visible ")
print("pages that deserve CTR/engagement review?")


Research question: Can search-performance and content signals identify visible 
pages that deserve CTR/engagement review?


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source:** the FlyRank ML Internship starter release, `data/raw/content_refresh_anonymized.csv` — 30,000 rows, one row per pseudonymized content item across 32 pseudonymized clients, trailing-90-day metrics. No client names, domains, URLs, or private queries are present in this file or in anything derived from it below.

**Unit of analysis:** one row = one content item (a page), as of the snapshot date.

**Excluded:** items with `impressions_90d == 0` or `avg_position == 0` (no meaningful position data — `avg_position = 0` means "no data", not rank zero) are dropped for the CTR-review lane, since a CTR judgement requires the page to actually be visible in search. This removes 1,205 of 30,000 rows, leaving 28,795 visible pages.

**Label:** `is_declining_label` is not a raw column — it is derived as `trend_direction == "down"` (16,262 of 30,000 rows, 54.2%). Because it is derived from `trend_direction`, both `trend_direction` and `trend_pct` are excluded from model features (see Methodology).

In [2]:
# Section 2 — Load and describe the data
import pandas as pd

url = "https://raw.githubusercontent.com/SUKRIT004/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
data = pd.read_csv(url)
data["is_declining_label"] = (data["trend_direction"] == "down").astype(int)

visible = data[(data["impressions_90d"] > 0) & (data["avg_position"] > 0)].copy()

print("Total rows:", len(data))
print("Visible rows (used for the CTR lane):", len(visible))
print("Excluded (not visible):", len(data) - len(visible))
print("Clients:", data["client_id"].nunique())
print("is_declining_label positive rate:", round(data["is_declining_label"].mean(), 4))


Total rows: 30000
Visible rows (used for the CTR lane): 28795
Excluded (not visible): 1205
Clients: 32
is_declining_label positive rate: 0.5421


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Baseline (Week 4, transparent rule):** `expected_ctr` = mean CTR for a page's `position_tier`; `ctr_gap` = `expected_ctr - ctr` (clipped at 0); `opportunity_score = ctr_gap * log1p(impressions_90d)`. No fitted weights — fully readable.

**Model (Week 5, Random Forest):** target `is_declining_label`; 25 features spanning traffic, engagement, content, and position signals (see feature list in the Results cell). **Excluded from features:** `is_declining_label` (the label itself), `trend_direction` and `trend_pct` (the label is derived from these — including them would leak the answer), and `content_id`/`client_id` (identifiers, not measurements).

**Validation design:** client-level holdout (`GroupShuffleSplit` grouped by `client_id`, 20% of clients held out, seed=42) so no client's pages appear in both train and test — a random row-level split would let the model memorize client-specific patterns and overstate its real-world skill. Train: 23,837 rows / 25 clients. Test: 6,163 rows / 7 clients.

**Leakage checks run:** (1) train-without-suspect test on `trend_direction`/`trend_pct` confirmed by exclusion rather than empirically re-added, since including a label-derived column would trivially leak; (2) confirmed `is_declining_label`, `trend_direction`, `trend_pct`, `content_id`, `client_id` are absent from the final feature list (asserted in code below); (3) base rate printed next to every metric.

**Reproducibility note (methodological finding, not swept under the rug):** an earlier run of the Week-5 notebook had cached Precision@50 = 0.82 for the model, but its data-loading cells were later cleared, making that number unreproducible. Rebuilding the notebook to run top-to-bottom on the same split and seed reproduces ROC-AUC ≈ 0.6145 (materially unchanged) but Precision@50 = 0.74 — a wider baseline advantage in the same direction. Tree-ensemble top-K metrics can be sensitive to small library-version and internal-ordering differences even with a fixed seed (flagged in `skills/training-honest-models/SKILL.md`). **0.74, not 0.82, is the number reported throughout this paper**, since it is the one this notebook can actually reproduce.

In [3]:
# Section 3 — Rebuild the baseline and the model (same logic as w04/w05, reproducible here)
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- Baseline ---
baseline = visible.copy()
expected_ctr_full = baseline.groupby("position_tier")["ctr"].mean().rename("expected_ctr")
baseline = baseline.join(expected_ctr_full, on="position_tier")
baseline["ctr_gap"] = baseline["expected_ctr"] - baseline["ctr"]
baseline["opportunity_score"] = baseline["ctr_gap"].clip(lower=0) * np.log1p(baseline["impressions_90d"])

# --- Model ---
target = "is_declining_label"
group = "client_id"
numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "search_volume", "competition", "cpc",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "age_tier",
    "freshness_tier", "position_tier", "impression_tier",
]
features = numeric_features + categorical_features

# Leakage assertion — the label and its source columns, and IDs, must never be features
forbidden = {"is_declining_label", "trend_direction", "trend_pct", "content_id", "client_id"}
assert not (forbidden & set(features)), f"Leakage: {forbidden & set(features)}"
print("Leakage check passed — none of", forbidden, "are in the feature list.")

X = data[features].copy()
y = data[target].astype(int)
groups = data[group]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("\nTrain rows:", len(X_train), "| Train clients:", data.iloc[train_idx][group].nunique())
print("Test rows:", len(X_test), "| Test clients:", data.iloc[test_idx][group].nunique())
print("Base rate (test):", round(y_test.mean(), 4))

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])
model = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=10,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
pipeline.fit(X_train, y_train)
test_probability = pipeline.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, test_probability)
print("\nROC-AUC:", round(roc_auc, 4))


Leakage check passed — none of {'client_id', 'is_declining_label', 'trend_direction', 'trend_pct', 'content_id'} are in the feature list.

Train rows: 23837 | Train clients: 25
Test rows: 6163 | Test clients: 7
Base rate (test): 0.511



ROC-AUC: 0.6145


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

On the held-out client split, the Week-4 transparent baseline reaches **Precision@50 = 0.84**; the Random Forest reaches **Precision@50 = 0.74** (ROC-AUC = 0.6145, base rate = 0.51). The simpler, fully-readable rule beats the more complex model on the metric that matters for this decision (which 50 pages to review first). The model's feature importances are shown for interpretability, not because the model won.

In [4]:
# Section 4 — Model vs baseline on the SAME held-out rows, plus feature importance
evaluation = data.iloc[test_idx].copy()
evaluation["model_score"] = test_probability
evaluation = evaluation.sort_values("model_score", ascending=False)
model_precision_at_50 = evaluation.head(50)[target].mean()

train_reference = data.iloc[train_idx].copy()
expected_ctr_train = train_reference.groupby("position_tier")["ctr"].mean()
evaluation["expected_ctr"] = evaluation["position_tier"].map(expected_ctr_train)
evaluation["ctr_gap"] = (evaluation["expected_ctr"] - evaluation["ctr"]).clip(lower=0)
evaluation["baseline_score"] = evaluation["ctr_gap"] * np.log1p(evaluation["impressions_90d"])
baseline_precision_at_50 = evaluation.sort_values("baseline_score", ascending=False).head(50)[target].mean()

comparison = pd.DataFrame({
    "method": ["Week-4 baseline (rule)", "Random Forest (model)"],
    "precision_at_50": [baseline_precision_at_50, model_precision_at_50],
    "roc_auc": [None, round(roc_auc, 4)],
})
print("=== MODEL VS BASELINE (same held-out split) ===")
display(comparison)
print("Base rate (test):", round(y_test.mean(), 4))
print("Baseline advantage:", round(baseline_precision_at_50 - model_precision_at_50, 4))

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_
importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
importance_df["feature"] = importance_df["feature"].str.replace(r"^(num__|cat__)", "", regex=True)
print("\nTop 10 feature importances:")
display(importance_df)


=== MODEL VS BASELINE (same held-out split) ===


,method,precision_at_50,roc_auc
0,Week-4 baseline (rule),0.84,NaN
1,Random Forest (model),0.74,0.6145


Base rate (test): 0.511
Baseline advantage: 0.1

Top 10 feature importances:


,feature,importance
0,impressions_90d,0.168449
1,content_age_days,0.119216
2,avg_position,0.117307
3,scroll_rate,0.047852
4,ctr,0.043144
5,clicks_90d,0.035556
6,search_volume,0.033338
7,position_tier_top_3,0.032860
8,pageviews_90d,0.030772
9,days_since_last_update,0.030454


## 5. Limitations

*What this work cannot claim.*

- **Not causal.** This is cross-sectional, one-snapshot data. Nothing here shows that editing a page *causes* CTR or rankings to change — only that certain pages *look* worth reviewing given current patterns.
- **Not a Google-algorithm model.** The model learns associations in one anonymized client portfolio's outcomes, not Google's ranking system.
- **Precision@50 ≠ precision everywhere.** Both methods were evaluated at K=50 on one held-out client split; performance at other K, or on a different client mix, is not shown here.
- **Small held-out set.** 7 held-out clients is a thin sample for the test side of a grouped split — the precision numbers could move with a different split seed. Directional conclusion (baseline ≥ model) held across two separate reruns of this notebook family (see the Methodology reproducibility note), which is reassuring but not a guarantee.
- **`ctr_gap` compares a page only to its position-tier average**, not to its specific query or competitive context — the Week-4 baseline notebook flags this explicitly as a known weakness.
- **Selection scope.** The queue only covers visible pages (impressions_90d > 0, avg_position > 0); pages with zero search visibility are excluded from this lane by definition, not because they don't matter.
- **Language used throughout:** observed, measured, associated, directional, decision-support — never "proves", "causes", or "predicts Google's algorithm".

In [5]:
# Section 5 has no computation — limitations are a written commitment, not a metric.
print("See markdown above for the full limitations statement.")


See markdown above for the full limitations statement.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The final ranked queue (built in `w07_action_playbook.ipynb`, loaded here from `work/outputs/action_playbook.csv`) combines the baseline opportunity score (60% weight, since it is the validated winner) with the model's decline-risk probability (40% weight, as a corroborating signal) into one `final_score`, with plain-language `reason_codes` on every row. Public-safe: no `content_id`/`client_id`, no client names, no URLs.

In [6]:
# Section 6 — Load and summarize the ranked action playbook
from pathlib import Path

# Resolve relative to the repo root regardless of the notebook's working directory
_repo_root = Path.cwd()
if not (_repo_root / "work" / "outputs").exists():
    _repo_root = Path.cwd().parent.parent  # work/notebooks -> repo root
playbook_path = _repo_root / "work" / "outputs" / "action_playbook.csv"
action_playbook = pd.read_csv(playbook_path)

print("Queue rows:", len(action_playbook))
print("\nSuggested-action mix:")
print(action_playbook["suggested_action"].value_counts())

print("\nTop 10 ranked recommendations:")
display(action_playbook.head(10)[[
    "rank", "content_type", "position_tier", "impressions_90d",
    "final_score", "reason_codes", "suggested_action",
]])


Queue rows: 28795

Suggested-action mix:
suggested_action
review_ctr                13591
review_ctr_and_content    10180
no_action                  3684
monitor_decline_risk       1340
Name: count, dtype: int64

Top 10 ranked recommendations:


,rank,content_type,position_tier,impressions_90d,final_score,reason_codes,suggested_action
0,1,keyword article,top_3,24784,82.551249,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
1,2,keyword article,top_3,24260,79.909026,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
2,3,keyword article,top_3,128068,79.772687,low_ctr_vs_position|high_exposure,review_ctr
3,4,keyword article,top_3,16512,79.430657,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
4,5,keyword article,top_3,8305,79.298571,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
5,6,keyword article,top_3,29747,79.244869,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
6,7,keyword article,top_3,26470,78.962968,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
7,8,keyword article,top_3,12053,77.914284,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
8,9,keyword article,top_3,19035,77.897208,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content
9,10,keyword article,top_3,10277,77.794851,low_ctr_vs_position|model_flags_decline_risk|h...,review_ctr_and_content


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Charts are generated by `scripts/build_capstone_artifacts.py` (single source of truth for every number and figure in this notebook and the paper) into `work/outputs/charts/`. Listed here for the record; the paper embeds copies in `docs/assets/charts/`.

In [7]:
# Section 7 — Confirm the paper's chart/table artifacts exist and match this run
import json as _json

charts_dir = _repo_root / "work" / "outputs" / "charts"
for f in sorted(charts_dir.glob("*.png")):
    print("chart:", f.name)

summary_path = _repo_root / "work" / "outputs" / "summary_stats.json"
with open(summary_path) as f:
    summary = _json.load(f)

print("\nHeadline numbers (work/outputs/summary_stats.json):")
for k in ["roc_auc", "model_precision_at_50", "baseline_precision_at_50", "base_rate_test",
          "train_rows", "test_rows", "train_clients", "test_clients", "playbook_rows"]:
    print(f"  {k}: {summary[k]}")


chart: action_mix.png
chart: ctr_by_position.png
chart: feature_importance.png
chart: opportunity_distribution.png
chart: precision_comparison.png

Headline numbers (work/outputs/summary_stats.json):
  roc_auc: 0.6145385259389705
  model_precision_at_50: 0.74
  baseline_precision_at_50: 0.84
  base_rate_test: 0.5109524582184002
  train_rows: 23837
  test_rows: 6163
  train_clients: 25
  test_clients: 7
  playbook_rows: 28795


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
